# Build a simple LLM application with chat models and prompt templates
https://python.langchain.com/docs/tutorials/llm_chain/

In [1]:
conda env list


# conda environments:
#
base                   /opt/anaconda3
HuggingFaceBook        /opt/anaconda3/envs/HuggingFaceBook
islp                   /opt/anaconda3/envs/islp
langsmith            * /opt/anaconda3/envs/langsmith
py313                  /opt/anaconda3/envs/py313
python-ds-ml-bootcamp   /opt/anaconda3/envs/python-ds-ml-bootcamp


Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import os
import jk_secrets

try:
    # load environment variables from .env file (requires `python-dotenv`)
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

os.environ["LANGSMITH_API_KEY"] = jk_secrets.LANGSMITH_API_KEY

os.environ["LANGSMITH_TRACING"] = "true"
if "LANGSMITH_API_KEY" not in os.environ:
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
        prompt="Enter your LangSmith API key (optional): "
    )

os.environ["LANGSMITH_PROJECT"] = "default"

if "LANGSMITH_PROJECT" not in os.environ:
    os.environ["LANGSMITH_PROJECT"] = getpass.getpass(
        prompt='Enter your LangSmith Project Name (default = "default"): '
    )
    if not os.environ.get("LANGSMITH_PROJECT"):
        os.environ["LANGSMITH_PROJECT"] = "default"


In [3]:
# pip install -qU "langchain[google-genai]"

In [4]:
import getpass
import os

os.environ["GOOGLE_API_KEY"] = os.getenv('GOOGLE_API_KEY')

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("Translate the following from English into Italian"),
    HumanMessage("hi!"),
]

model.invoke(messages)

AIMessage(content='Ciao!', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--61bca927-7eb4-45d9-8a00-2ae08cc950ea-0', usage_metadata={'input_tokens': 10, 'output_tokens': 36, 'total_tokens': 46, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 34}})

### Tracing projects: 

https://smith.langchain.com/o/56929930-fae7-411c-bdc0-39dedbe98e07/projects

In [6]:
model.invoke("Hello")

model.invoke([{"role": "user", "content": "Hello"}])

model.invoke([HumanMessage("Hello")])

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--451068f1-d6b1-44cc-b927-77cd49cc8c41-0', usage_metadata={'input_tokens': 2, 'output_tokens': 157, 'total_tokens': 159, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 148}})

In [7]:
for token in model.stream(messages):
    print(token.content, end="|")

Ciao!|

In [8]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)